# A Multi-Source Dataset for Mwense Town Council

**CSC 4792: Data Mining and Warehousing**
**Project team:** Group 33
**Assigned council:** Mwense Town Council
**Official website:** https://www.mwensecouncil.gov.zm/
**Submission deadline:** 14 September 2026

This notebook documents the collection, cleaning, analysis, and export of Mwense Town Council's public records. Running it from start to finish reproduces the Kaggle dataset in `outputs/` as pipe-separated CSV files.

## 1. Project Purpose and Scope

Council information on projects, budgets, plans, and administration is distributed across web pages and PDF documents. This project consolidates those sources into a single structured dataset.

The dataset covers CDF projects, approved budgets, LGEF utilisation, locally generated revenue, Integrated Development Plans, wards, administrative structures, and council reports or resolutions, subject to what the sources publish. Fields without source evidence are left missing rather than estimated.

## 2. Research Questions

1. Which development projects and priorities are documented for Mwense Town Council?
2. How are records distributed across CDF, education, health, water, sanitation, roads, and other sectors?
3. Which budget allocations, expenditures, revenues, or LGEF figures are publicly reported?
4. Which wards, constituencies, and administrative units are represented?
5. What project statuses, dates, beneficiaries, and implementation details are available?
6. What gaps and inconsistencies limit reuse of the data?

Section 6 addresses each question with tables and charts, and reports gaps explicitly.

In [ ]:
# Import libraries required for extraction, cleaning, analysis, and visualisation
import re
import subprocess
import sys
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)

print("Libraries imported successfully.")

In [ ]:
# Dependency check: install anything missing so fresh kernels and Colab work without manual setup
import importlib.util as _ilu
_REQUIRED = {"pandas": "pandas", "numpy": "numpy", "requests": "requests", "bs4": "beautifulsoup4", "matplotlib": "matplotlib", "seaborn": "seaborn", "pypdf": "pypdf"}
_missing = [pip for mod, pip in _REQUIRED.items() if _ilu.find_spec(mod) is None]
if _missing:
    print(f"Installing missing packages: {_missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", *_missing], check=True)
    print("Install complete. If the import cell above ran before this, rerun it once.")
else:
    print("All required packages available.")

In [ ]:
# Define reproducible project configuration
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "raw_sources"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = PROJECT_ROOT / "figures"
for directory in (RAW_DIR, OUTPUT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

COUNCIL_NAME = "Mwense Town Council"
BASE_URL = "https://www.mwensecouncil.gov.zm/"
ACCESS_DATE = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
REQUEST_TIMEOUT = 30
HEADERS = {"User-Agent": "CSC-4792-Group-33-research/1.0"}

print(f"Project: {COUNCIL_NAME}")
print(f"Base URL: {BASE_URL}")
print(f"Access date and time: {ACCESS_DATE}")

## 3. Data Collection Methodology

Collection proceeds in four stages: identify relevant pages and documents on the council website, extract text while retaining source metadata, clean and normalise records without discarding evidence, then analyse and export.

Each record retains `source_id`, `source_url`, `source_title`, `access_date`, and `extraction_notes` to support verification and the Data in Brief methodology section. Section 3 handles acquisition, Section 4 defines the schema, Section 5 cleaning, and Section 6 analysis.

## 3.1 Reproducible Source Acquisition

Acquisition is implemented in standalone scripts invoked from the notebook to keep the workflow reproducible. The download cell calls `scripts/crawl_mwense_sources.py` for the site manifest and `scripts/download_mwense_sources.py` for the four core PDFs into `raw_sources/`. Both skip files already present and up to date; `--force` refreshes them. The extraction cell calls `scripts/extract_mwense_pdfs.py` to produce reviewable text in `raw_sources/extracted_text/`. The listing cell reports the PDFs and text files present with file sizes.

`RUN_SITE_CRAWL`, `RUN_SOURCE_DOWNLOAD`, and `RUN_PDF_EXTRACTION` default to `False` to reuse local files; set them to `True` to refetch. The crawler requires the `--insecure` flag in this environment.

In [ ]:
# Download the official source PDFs through the project scripts.
# Reproducibility: download_mwense_sources.py skips files that already exist
# and are up-to-date (local size == remote Content-Length via HEAD), so a
# top-to-bottom re-run is fast. Use --force to refresh everything.
# Crawler requires --insecure on this host (council TLS chain fails verify).
# Set RUN_SITE_CRAWL = True to scan categories and refresh the manifest.
# Set RUN_SOURCE_DOWNLOAD = True to refresh the fixed set of core PDFs.
RUN_SITE_CRAWL = False
RUN_SOURCE_DOWNLOAD = False
CRAWL_SCRIPT = PROJECT_ROOT / "scripts" / "crawl_mwense_sources.py"
DOWNLOAD_SCRIPT = PROJECT_ROOT / "scripts" / "download_mwense_sources.py"

if RUN_SITE_CRAWL:
    subprocess.run([sys.executable, str(CRAWL_SCRIPT), "--max-pages", "100", "--insecure"], check=True)
else:
    print("Site crawl skipped. Set RUN_SITE_CRAWL = True to scan the council source categories.")

if RUN_SOURCE_DOWNLOAD:
    subprocess.run([sys.executable, str(DOWNLOAD_SCRIPT)], check=True)
else:
    print("Fixed-source download skipped (existing up-to-date PDFs reused). Set RUN_SOURCE_DOWNLOAD = True to check/refresh core PDFs.")


In [ ]:
# Extract text from downloaded PDFs through the project script.
# extract_mwense_pdfs.py skips extraction when the .txt is newer than its .pdf,
# so re-runs are fast. Use --force to re-extract everything.
# Note: the 2024 CDF PDF is a scanned image (see Section 3.2) - pypdf yields
# almost no text, so 2024 rows below are manually verified from the PDF visual
# with source wording preserved in raw_text (see provenance cell).
RUN_PDF_EXTRACTION = False
EXTRACTION_SCRIPT = PROJECT_ROOT / "scripts" / "extract_mwense_pdfs.py"

if RUN_PDF_EXTRACTION:
    subprocess.run([sys.executable, str(EXTRACTION_SCRIPT)], check=True)
else:
    print("PDF extraction skipped (existing extracted text reused). Set RUN_PDF_EXTRACTION = True after downloading sources.")


In [ ]:
# Report the local source artifacts available to the notebook.
pdf_files = sorted(RAW_DIR.glob("*.pdf"))
text_files = sorted((RAW_DIR / "extracted_text").glob("*.txt")) if (RAW_DIR / "extracted_text").exists() else []

print("Downloaded PDF files:")
for path in pdf_files:
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")
print("\nExtracted text files:")
for path in text_files:
    size = path.stat().st_size
    flag = "  <-- WARNING: near-empty, likely scanned image (see Section 3.2)" if size < 100 else ""
    print(f"- {path.relative_to(PROJECT_ROOT)} ({size:,} bytes){flag}")
if not pdf_files:
    print("No local PDFs found. Run the source-download cell before final submission.")


## 3.2 Source-text provenance and 2024 scanned-PDF limitation

The extracted text for `mwense-2024-approved-cdf-projects.pdf` contains page headers only (approximately 32 bytes). The source PDF is a scanned image, so `pypdf` cannot recover its tables.

The 2024 project names and `budget_amount_zmw` values were verified against the PDF and transcribed, with source wording retained in `raw_text`. The apparent amount anomaly and the Mambilima heading listing 13 rows under a heading of 12 are flagged in `extraction_notes`. The 2025 approved lists publish no monetary amounts, so the corresponding budget fields remain missing. No values were inferred.

In [ ]:
# Maintain a traceable inventory of the official sources used in this review dataset.
source_inventory = pd.DataFrame([
    {
        "source_id": "SRC001",
        "source_url": BASE_URL,
        "source_title": "Mwense Town Council official website and navigation pages",
        "source_type": "website",
        "topic": "council profile, departments, CDF, publications, and news",
        "publication_date": pd.NaT,
        "access_date": ACCESS_DATE,
        "notes": "Homepage and linked pages; accessed during project review."
    },
    {
        "source_id": "SRC002",
        "source_url": "https://www.mwensecouncil.gov.zm/?page_id=3026",
        "source_title": "List of approved projects page",
        "source_type": "webpage",
        "topic": "approved CDF and capital projects",
        "publication_date": pd.NaT,
        "access_date": ACCESS_DATE,
        "notes": "Page links to the 2024 and 2025 project PDFs."
    },
    {
        "source_id": "SRC003",
        "source_url": "https://www.mwensecouncil.gov.zm/wp-content/uploads/2024/09/IDP-MWENSE.pdf",
        "source_title": "Integrated Development Plan 2024-2034",
        "source_type": "pdf",
        "topic": "planning, sectors, wards, committees, and development priorities",
        "publication_date": "2024-09-01",
        "access_date": ACCESS_DATE,
        "notes": "Official IDP PDF; publication date inferred from upload path and document title."
    },
    {
        "source_id": "SRC004",
        "source_url": "https://www.mwensecouncil.gov.zm/wp-content/uploads/2025/06/LIST-OF-2024-APPROVED-CDF-PROJECTS.pdf",
        "source_title": "2024 approved CDF community projects",
        "source_type": "pdf",
        "topic": "2024 Mwense Central and Mambilima CDF projects",
        "publication_date": "2024-01-30",
        "access_date": ACCESS_DATE,
        "notes": "Official approval letter. The Mambilima table contains 13 rows although the heading says 12."
    },
    {
        "source_id": "SRC005",
        "source_url": "https://www.mwensecouncil.gov.zm/wp-content/uploads/2025/06/LIST-OF-2025-APPROVED-PROJECTS.pdf",
        "source_title": "2025 approved capital and CDF projects",
        "source_type": "pdf",
        "topic": "2025 capital projects, CDF projects, wards, sectors, and resolutions",
        "publication_date": "2025-06-13",
        "access_date": ACCESS_DATE,
        "notes": "Official council project list containing approved and differed decisions."
    },
    {
        "source_id": "SRC006",
        "source_url": "https://www.mwensecouncil.gov.zm/wp-content/uploads/2023/07/2022-CDF-GUIDELINES.pdf",
        "source_title": "CDF Guidelines, February 2022",
        "source_type": "pdf",
        "topic": "CDF governance, disbursement, utilisation, and accountability framework",
        "publication_date": "2022-02-01",
        "access_date": ACCESS_DATE,
        "notes": "National guideline linked from the council website; used as contextual documentation."
    }
])

source_inventory["publication_date"] = pd.to_datetime(source_inventory["publication_date"], errors="coerce")
source_inventory

In [ ]:
# Define the final record schema before constructing project rows.
record_columns = [
    "record_id", "source_id", "source_url", "source_title", "source_type",
    "publication_date", "access_date", "record_category", "project_name",
    "project_description", "sector", "ward", "community", "constituency",
    "institution", "funding_source", "budget_amount_zmw", "amount_disbursed_zmw",
    "amount_spent_zmw", "project_status", "start_date", "completion_date",
    "beneficiaries", "raw_text", "extraction_notes"
]
print(f"Final schema contains {len(record_columns)} fields.")

In [ ]:
# Build one row per verified project from the official 2024 and 2025 project lists.
# Financial amounts are populated only where the source explicitly reports an allocation.
records = []

def add_record(source_id, category, name, sector, constituency="", ward="", status="Approved",
               funding_source="", amount=np.nan, description="", notes=""):
    source = source_inventory.loc[source_inventory["source_id"] == source_id].iloc[0]
    records.append({
        "record_id": f"REC{len(records) + 1:03d}",
        "source_id": source_id,
        "source_url": source["source_url"],
        "source_title": source["source_title"],
        "source_type": source["source_type"],
        "publication_date": source["publication_date"],
        "access_date": source["access_date"],
        "record_category": category,
        "project_name": name,
        "project_description": description,
        "sector": sector,
        "ward": ward,
        "community": "",
        "constituency": constituency,
        "institution": "",
        "funding_source": funding_source,
        "budget_amount_zmw": amount,
        "amount_disbursed_zmw": np.nan,
        "amount_spent_zmw": np.nan,
        "project_status": status,
        "start_date": pd.NaT,
        "completion_date": pd.NaT,
        "beneficiaries": "",
        "raw_text": name,
        "extraction_notes": notes
    })

# 2025 basic capital grant and own-source projects.
capital_2025 = [
    ("Construction of a modern bus station", "Basic capital grant"),
    ("Procurement of a skip truck", "Basic capital grant"),
    ("Procurement of five skip bins", "Basic capital grant"),
    ("Procurement of a fire engine", "Basic capital grant"),
    ("Maintenance of township roads", "Motor Licensing Fund"),
    ("Drainage clearing and desilting", "Motor Licensing Fund"),
    ("Installation and maintenance of 20 street lights", "Motor Licensing Fund"),
    ("Construction of crossing points", "Motor Licensing Fund"),
    ("Construction of chalets at the council guest house", "Locally generated revenue/own source revenue")
]
for name, funding in capital_2025:
    add_record("SRC005", "capital_project", name, "Infrastructure", funding_source=funding,
               notes="2025 approved capital project list.")

# 2025 approved CDF projects: Mwense Constituency.
mwense_2025 = [
    ("Procurement of a 12-body mortuary unit for Mwense District Hospital", "Health", "All wards", "Supply and installation of a 12-body cadaver unit and backup generator"),
    ("Contribution towards holding cells at Mwense Police Station", "Security", "All wards", "Adult male, adult female, female juvenile, and male juvenile cells"),
    ("Construction of a 1 by 3 CRB block at Mwense Primary School", "Education", "Kasengu", "1 by 3 classroom block with 75 desks"),
    ("Rehabilitation of piped water network in Lubunda and Kankomba", "Water and sanitation", "Katiti", "Two transformers and rehabilitation of the current network"),
    ("Construction of male and female hostels at Lukwesa Secondary School", "Education", "Pebekabesa", "Two hostels and 88 four-inch mattresses"),
    ("Construction of a 1 by 3 CRB and piped water at Kawama Secondary School", "Education", "Nkanga", "Classroom block, 75 desks, and borehole equipment"),
    ("Construction and extension of Lwamfwe mini-water scheme", "Water and sanitation", "Kaombe", "40,000-litre tanks, transformer, and network extension"),
    ("Rural electrification of households", "Energy", "Kapamba, Luche, Kapela", "Electrification of 600 households"),
    ("Procurement of 600 mattresses for Mwense Secondary School", "Education", "Mwense", "Supply and delivery of 600 four-inch mattresses"),
    ("Construction of a 1 by 2 science laboratory at Kasonge Secondary School", "Education", "James Chiwasha", "Phase 1 science laboratory")
]
for name, sector, ward, description in mwense_2025:
    add_record("SRC005", "cdf_project", name, sector, constituency="Mwense", ward=ward,
               description=description, notes="2025 approved CDF project list.")

# 2025 approved CDF projects: Mambilima Constituency.
mambilima_2025 = [
    ("Construction of a chief's palace", "Infrastructure", "All wards"),
    ("Purchase of a motorbike", "Transport", "All wards"),
    ("Construction of police cells", "Home Affairs", "All wards"),
    ("Construction of staff house, ablution block, and water scheme at Sepe Health Post", "Health", "Michelo"),
    ("Construction of mortuary shelter at Katuta Clinic", "Health", "Mpasa"),
    ("Construction of a 40,000-litre water scheme at Kabunda Village", "Water and sanitation", "Chibembe"),
    ("Construction of a 1 by 3 CRB at Kabila Primary School", "Education", "Musonda"),
    ("Construction of a water scheme at Chula Village", "Water and sanitation", "Musonda"),
    ("Construction of a 1 by 3 CRB at Mutima Secondary School", "Education", "Munwa"),
    ("Construction of an ablution block and two staff houses at Chebele", "Health", "Kalanga"),
    ("Drilling of boreholes with Afridev hand pumps at Ndombi and Mwendango villages", "Water and sanitation", "Nsomfi")
]
for name, sector, ward in mambilima_2025:
    add_record("SRC005", "cdf_project", name, sector, constituency="Mambilima", ward=ward,
               notes="2025 approved CDF project list.")

# 2024 approved CDF projects: Mwense Central Constituency.
mwense_2024 = [
    ("Construction of Chiefs Palace", "Infrastructure", 1000000.00),
    ("Procurement of an ambulance", "Health", 2700000.00),
    ("Procurement of three motorbikes", "Transport", 230000.00),
    ("Grading of feeder roads", "Roads", 800000.00),
    ("Contribution to construct a female prison cell", "Infrastructure", 450000.00),
    ("Admission ward at Mwense Stage Two", "Health", 800000.00),
    ("Construction of four maternity annexes with water", "Health", 5000000.00),
    ("Piped water at Nsemba Village", "Water", 725000.00),
    ("Completion of Muposhi RHC post with water and electricity", "Health", np.nan),
    ("Construction of 1 by 3 classroom block, ablution block, and desks", "Education", 2509200.00),
    ("Construction of staff house at Kapamba RHC", "Health", 500000.00),
    ("Construction of water scheme at Tela Village", "Water", 725000.00),
    ("Rehabilitation of Kakusa water scheme", "Water", 200000.00)
]
for name, sector, amount in mwense_2024:
    note = "2024 approval letter; source contains an apparent OCR/typographical amount issue." if pd.isna(amount) else "2024 approval letter."
    add_record("SRC004", "cdf_project", name, sector, constituency="Mwense Central",
               amount=amount, notes=note)

# 2024 approved CDF projects: Mambilima Constituency.
mambilima_2024 = [
    ("Construction of Chiefs Palace", "Infrastructure", 1000000.00),
    ("Procurement of an ambulance", "Health", 2700000.00),
    ("Procurement of three motorbikes", "Transport", 345000.00),
    ("Grading of feeder roads", "Roads", 1000000.00),
    ("Contribution to construct a female prison cell", "Infrastructure", 450000.00),
    ("Extension of water lines at Chalwe Primary School and village", "Water", 950000.00),
    ("Construction of maternity annex at Kabangila wing", "Health", 756200.41),
    ("Piped water from Nkomba to Mini Hospital", "Health", 1850000.00),
    ("Construction of drainage from Kashiba to Nsomfi stream", "Roads", 2070000.00),
    ("Construction of maternity annex, equipment, and staff house", "Health", 1500000.00),
    ("Construction of drainage from checkpoint to Shichama", "Roads", 2618000.00),
    ("Construction of police post and staff house at a new site", "Infrastructure", 1350000.00),
    ("Procurement of equipment for seven GRZ facilities", "Health and Education", 1693025.00)
]
for name, sector, amount in mambilima_2024:
    add_record("SRC004", "cdf_project", name, sector, constituency="Mambilima",
               amount=amount, notes="2024 approval letter; table contains 13 projects although the heading says 12.")

raw_df = pd.DataFrame(records, columns=record_columns)
print(f"Loaded {len(raw_df)} verified project records.")
print(raw_df.groupby(["source_id", "record_category"]).size().rename("records"))
raw_df.head()

In [ ]:
# Add non-project records required to represent the IDP and council administration.
add_record(
    "SRC003", "idp_record", "Integrated Development Plan 2024-2034", "Planning",
    description="Ten-year participatory plan covering water and sanitation, energy, housing and community amenities, forestry, roads, agriculture, fisheries and livestock, social protection, education, and health.",
    notes="IDP states that Ward Development Committees and community members contributed to preparation."
)
add_record(
    "SRC003", "administrative_record", "Mwense IDP participatory planning structure", "Administration",
    description="Planning involved the council, Ward Development Committees, Constituency Development Fund Committees, traditional leadership, government departments, and community members.",
    notes="Extracted from the IDP acknowledgements and planning-team sections."
)
add_record(
    "SRC001", "administrative_record", "Mwense Town Council profile", "Administration",
    description="Mwense Town Council is located in Mwense District, Luapula Province, Zambia.",
    notes="Extracted from the official council website homepage."
)
raw_df = pd.DataFrame(records, columns=record_columns)
print(f"Dataset now contains {len(raw_df)} project, IDP, and administrative records.")

## 4. Dataset Schema

The schema defines one row per council record with 25 columns, separating record content from provenance. The following code cells declare the column list, construct 56 project rows from the 2024 and 2025 approved lists (9 capital and 47 CDF projects), append 3 IDP and administrative records, and display the resulting row counts and leading rows.

In [ ]:
# Inspect the populated raw-record table before applying cleaning rules.
if "raw_df" not in globals() or raw_df.empty:
    raise ValueError("raw_df is empty. Run the source inventory, schema, and record-construction cells first.")

print(f"Raw records: {len(raw_df)}")
print(f"Raw columns: {len(raw_df.columns)}")
raw_df.head()

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return pd.NA
    value = re.sub(r"\\s+", " ", str(value)).strip()
    return value if value else pd.NA

def clean_currency(value):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    cleaned = re.sub(r"[^0-9.]", "", str(value))
    return pd.to_numeric(cleaned, errors="coerce")

cleaned_df = raw_df.copy()
text_columns = cleaned_df.select_dtypes(include="object").columns
for column in text_columns:
    cleaned_df[column] = cleaned_df[column].map(clean_text)

for column in ["budget_amount_zmw", "amount_disbursed_zmw", "amount_spent_zmw"]:
    cleaned_df[column] = cleaned_df[column].map(clean_currency)

for column in ["publication_date", "access_date", "start_date", "completion_date"]:
    cleaned_df[column] = pd.to_datetime(cleaned_df[column], errors="coerce")

# Controlled-vocabulary normalisation (documented, no values invented):
# - "Water" -> "Water and sanitation" (same council usage across years).
# - "Home Affairs" police-cells -> "Security" (matches 2025 Mwense holding-cells label).
SECTOR_MAP = {"Water": "Water and sanitation", "Home Affairs": "Security"}
cleaned_df["sector"] = cleaned_df["sector"].replace(SECTOR_MAP)
# Ward: preserve provenance - empty after clean_text means source did not report it.
# Use "Not reported" for analysis so missingness is explicit (vs district-wide "All wards").
cleaned_df["ward"] = cleaned_df["ward"].fillna("Not reported")

print("Cleaning rules + sector/ward normalisation applied to the working copy.")
print(f"Sectors now: {sorted(cleaned_df['sector'].dropna().unique().tolist())}")


## 5. Cleaning and Quality Assurance

Cleaning operates on a working copy (`cleaned_df`); the raw table (`raw_df`) is preserved. The cleaning cell standardises whitespace, parses monetary values and dates, harmonises sector labels (`Water` to `Water and sanitation`; police cells under `Home Affairs` to `Security`), and assigns `Not reported` to wards absent from the source to make coverage gaps visible.

The quality-control cell verifies unique record identifiers, complete source references, absence of duplicate rows and negative financial values, and reports missingness by column. Missing values indicate absence from the source, not zero.

In [ ]:
# Produce a concise quality-control report
quality_report = {
    "rows": len(cleaned_df),
    "columns": len(cleaned_df.columns),
    "duplicate_rows": int(cleaned_df.duplicated().sum()),
    "duplicate_record_ids": int(cleaned_df["record_id"].duplicated().sum()),
    "missing_source_ids": int(cleaned_df["source_id"].isna().sum()),
    "missing_project_categories": int(cleaned_df["record_category"].isna().sum()),
    "negative_financial_values": int((cleaned_df[["budget_amount_zmw", "amount_disbursed_zmw", "amount_spent_zmw"]] < 0).sum().sum()),
}

for item, value in quality_report.items():
    print(f"{item}: {value}")

missing_summary = (
    cleaned_df.isna().sum()
    .rename("missing")
    .to_frame()
)
missing_summary["percentage"] = 100 * missing_summary["missing"] / max(len(cleaned_df), 1)
missing_summary.sort_values("missing", ascending=False).head(15)

## 6. Descriptive and Thematic Analysis

Analysis is descriptive. It summarises reported records without treating absent information as evidence of absence. Financial aggregates are computed only for 2024 CDF rows with compatible ZMW values; 2025 lists provide no amounts for aggregation.

The subsections below present overview counts (6.1), finance coverage (6.2), IDP and administrative coverage (6.3), and supporting visualisations (6.4).

In [ ]:
# Dataset overview tables
print("Records by category")
display(cleaned_df["record_category"].value_counts(dropna=False).rename_axis("category").to_frame("records"))

print("Records by sector")
display(cleaned_df["sector"].value_counts(dropna=False).rename_axis("sector").to_frame("records"))

print("Records by project status")
display(cleaned_df["project_status"].value_counts(dropna=False).rename_axis("status").to_frame("records"))

### 6.1 CDF, Projects, Budgets, LGEF, and Revenue

Tables report records by category, sector, and project status. The accompanying bar charts visualise records by sector and by status, saved to `figures/records-by-sector-and-status.png`.

Sector counts reflect publication coverage rather than relative expenditure.

In [ ]:
# Generate evidence-based plots when records are available
if len(cleaned_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    cleaned_df["sector"].value_counts(dropna=False).plot(kind="bar", ax=axes[0], color="#2f6f73")
    axes[0].set_title("Council records by sector")
    axes[0].set_xlabel("Sector")
    axes[0].set_ylabel("Number of records")
    axes[0].tick_params(axis="x", rotation=45)

    cleaned_df["project_status"].value_counts(dropna=False).plot(kind="bar", ax=axes[1], color="#c76b3c")
    axes[1].set_title("Project status distribution")
    axes[1].set_xlabel("Status")
    axes[1].set_ylabel("Number of records")
    axes[1].tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "records-by-sector-and-status.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No records are available yet. Add verified Mwense sources before generating plots.")

### 6.2 Budget, LGEF, and local-revenue coverage

The finance cell reports the number of rows with `budget_amount_zmw`, aggregates 2024 CDF amounts by constituency and record category, and tabulates `funding_source` values. It documents the resulting gaps: disbursed and spent amounts are entirely absent, LGEF figures do not appear in the inventoried sources, and capital revenue is present only as funding labels without amounts.

In [ ]:
# Finance coverage: CDF/budgets/LGEF/local revenue - reported amounts only, gaps stated
print(f"Rows with budget_amount_zmw: {cleaned_df['budget_amount_zmw'].notna().sum()} / {len(cleaned_df)}")
print("Budget rows exist only in the 2024 CDF approval letter; 2025 lists carry no amounts.")
display(cleaned_df[cleaned_df['budget_amount_zmw'].notna()].groupby('constituency', dropna=False)['budget_amount_zmw'].agg(records='count', total_zmw='sum').round(2).rename_axis('constituency'))
display(cleaned_df[cleaned_df['budget_amount_zmw'].notna()].groupby('record_category')['budget_amount_zmw'].agg(records='count', total_zmw='sum').round(2).rename_axis('record_category'))
display(cleaned_df['funding_source'].value_counts(dropna=False).rename_axis('funding_source').to_frame('records'))
print("Disbursed/spent: 100% missing - no inventoried source reports them.")
print("LGEF utilisation: 0 records in SRC001-SRC006.")
print("Local revenue: funding labels on 9 capital rows only, without amounts (see table above).")

### 6.3 IDP, WDC, and resolutions coverage

The IDP cell loads `clean_data/raw_idp_unclean.csv` (359 parsed rows) for context and lists the 3 IDP and administrative records in the main dataset. Ward Development Committees are referenced in the IDP without a published per-ward membership list, and 2025 differed decisions are noted without itemised detail; consequently only `Approved` records are retained.

In [ ]:
# IDP / WDC / resolutions coverage - 359-row parsed file plus 3 main-table rows
from pathlib import Path as _P
_idp_path = _P("clean_data/raw_idp_unclean.csv")
if _idp_path.exists():
    idp_raw = pd.read_csv(_idp_path, sep="|")
    print(f"IDP parsed rows: {len(idp_raw)}")
    type_col = next((c for c in idp_raw.columns if c.lower() in ("type", "record_type", "category")), idp_raw.columns[1])
    display(idp_raw[type_col].value_counts().head(10).rename_axis(type_col).to_frame('records'))
    print("IDP file columns:", list(idp_raw.columns))
else:
    print("clean_data/raw_idp_unclean.csv not found - commit clean_data/ for this cell to run.")
display(cleaned_df[cleaned_df['record_category'].isin(['idp_record', 'administrative_record'])][['record_id', 'project_name', 'sector']])
print("WDC: referenced in the IDP without a published per-ward membership list.")
print("Resolutions: 2025 differed decisions noted without itemised detail; only Approved rows retained.")

### 6.4 Finance, ward, and missingness visualisations

The visualisation cell produces three panels saved to `figures/finance-ward-missingness.png`: budget availability by constituency, ward frequency against `Not reported`, and missingness across the sparsest fields.

In [ ]:
# Additional evidence plots: finance, ward coverage, missingness
import matplotlib.pyplot as _plt
fig, axes = _plt.subplots(1, 3, figsize=(18, 5))

# 1. Budget availability by constituency (reported vs not reported)
_budget = cleaned_df.copy()
_budget['has_budget'] = _budget['budget_amount_zmw'].notna().map({True: 'Amount reported', False: 'Not reported'})
pd.crosstab(_budget['constituency'].fillna('(no constituency)'), _budget['has_budget']).plot(kind='bar', stacked=True, ax=axes[0], color=['#2f6f73','#c76b3c'])
axes[0].set_title('Budget reporting by constituency')
axes[0].set_xlabel('Constituency')
axes[0].set_ylabel('Records')
axes[0].tick_params(axis='x', rotation=30)

# 2. Ward coverage (top 8 + Not reported)
cleaned_df['ward'].fillna('Not reported').value_counts().head(8).plot(kind='barh', ax=axes[1], color='#4a7c59')
axes[1].set_title('Top wards (rest = long tail / Not reported)')
axes[1].set_xlabel('Records')

# 3. Missingness by major field
(cleaned_df.isna().sum().sort_values(ascending=False).head(10) / len(cleaned_df) * 100).plot(kind='bar', ax=axes[2], color='#6a6a6a')
axes[2].set_title('Missingness % (top 10 fields)')
axes[2].set_ylabel('% missing')
axes[2].tick_params(axis='x', rotation=45)

_plt.tight_layout()
_plt.savefig(FIGURE_DIR / "finance-ward-missingness.png", dpi=150, bbox_inches="tight")
_plt.show()
print("Saved figures/finance-ward-missingness.png")


## 7. Limitations and Reuse Value

Limitations: ward is `Not reported` in 38 of 59 records; community, institution, beneficiary, date, disbursed, and spent fields are fully absent; funding source is absent in 50 records, reflecting CDF rows without a stated funder; all statuses are `Approved`, so the dataset captures approvals rather than implementation; 12 legacy `content-*` section URLs return 404; and the 2024 PDF required manual transcription.

The dataset supports analysis of local-government transparency, CDF approval patterns, public-finance reporting, service-delivery coverage, and reuse in data-mining studies where source traceability is required.

## 8. Data Dictionary

25 columns as exported. Blank = not in the source or not applicable, never zero-filled.

| Field | Description | Type or unit | Missing-value meaning |
|---|---|---|---|
| `record_id` | Unique record identifier | String, `REC001`... | Must not be missing |
| `source_id` | Link to source inventory | String, `SRC001`... | Must be corrected if missing |
| `source_url` | URL where evidence was obtained | String, URL | URL unavailable |
| `source_title` | Human-readable source title | String | Title not recorded |
| `source_type` | `website`, `webpage`, or `pdf` | Categorical | Type not classified |
| `publication_date` | Source publication date | Date | Not stated in source |
| `access_date` | Date and time source was accessed | Datetime (`YYYY-MM-DD HH:MM`) | Must not be missing |
| `record_category` | `cdf_project`, `capital_project`, `idp_record`, `administrative_record` | Categorical | Must not be missing |
| `project_name` | Project or record name | String | Not reported / not applicable |
| `project_description` | Detail from source | String | Not reported |
| `sector` | Normalised sector (Infrastructure, Health, Education, Water and sanitation, Roads, Transport, Energy, Security, Administration, Planning, Health and Education) | Categorical | Not reported |
| `ward` | Ward, `All wards`, or `Not reported` | String | Source did not specify ward |
| `community` | Community/village | String | Not reported (100% in current sources) |
| `constituency` | `Mwense`, `Mambilima`, `Mwense Central`, or missing | String | Not applicable (e.g. capital/IDP rows) |
| `institution` | School/clinic/market name | String | Not separately stated (100% in current sources) |
| `funding_source` | Reported funder | String | Not reported (84.7% - mostly CDF rows without explicit funder line) |
| `budget_amount_zmw` | Reported budget/allocation | Numeric, ZMW | Not reported (57.6% - all 2025 rows, 1 of 2024 rows) |
| `amount_disbursed_zmw` | Reported disbursed | Numeric, ZMW | Not reported (100% - no source states it) |
| `amount_spent_zmw` | Reported spent | Numeric, ZMW | Not reported (100%) |
| `project_status` | Reported status | Categorical, `Approved` | Not reported |
| `start_date` | Start date | Date | Not reported (100%) |
| `completion_date` | Completion date | Date | Not reported (100%) |
| `beneficiaries` | Beneficiaries | String | Not reported (100%) |
| `raw_text` | Source wording for verification | String | Extraction produced no text |
| `extraction_notes` | Interpretation/quality notes | String | No additional notes |

In [ ]:
# Export the cleaned dataset and source inventory using the assignment's required pipe separator.
output_path = OUTPUT_DIR / "db-unza26-csc4792-mwense-town-council-records.csv"
sources_output_path = OUTPUT_DIR / "db-unza26-csc4792-mwense-town-council-sources.csv"
cleaned_df.to_csv(output_path, sep="|", index=False)
source_inventory.to_csv(sources_output_path, sep="|", index=False)

print(f"Exported {len(cleaned_df)} records to {output_path}")
print(f"Exported {len(source_inventory)} source records to {sources_output_path}")
print("Separator: pipe (|)")
print(f"Records filename compliant: {output_path.name.startswith('db-unza26-csc4792-')}")
print(f"Sources filename compliant: {sources_output_path.name.startswith('db-unza26-csc4792-')}")

# Kaggle-ready subsets with identical 25-column schema (no schema drift).
cdf_only = cleaned_df[cleaned_df["record_category"] == "cdf_project"]
cdf_path = OUTPUT_DIR / "db-unza26-csc4792-mwense-town-council-cdf-projects.csv"
cdf_only.to_csv(cdf_path, sep="|", index=False)
print(f"Exported {len(cdf_only)} CDF rows to {cdf_path}")


## 9. Submission Checklist

- Group 33 and Mwense Town Council identified throughout.
- All sources recorded in the inventory with URLs and access dates.
- Notebook executes top to bottom without undisclosed manual steps.
- Cleaning, missing-value, and normalisation decisions documented in Section 5.
- CDF, budgets, LGEF, revenue, IDP, ward, administrative, and resolution evidence addressed or recorded as absent.
- Exports in `outputs/` follow `db-unza26-csc4792-[description].csv` with `|` separation.
- Data dictionary matches the 25 exported columns.
- Notebook and scripts committed to GitHub with descriptive messages; `lighton.phiri@gmail.com` added.
- Kaggle documentation follows the class exemplar; Data in Brief paper addresses method, structure, limitations, and reuse value.
- Kaggle link, GitHub link, notebook, and paper prepared for Moodle submission by Monday 14th.